# `data-conduit` TODO / v1 Roadmap

This notebook is the short version.

It is meant to answer four questions quickly:

1. What is the project like right now?
2. What is wrong or missing?
3. What should the final structure look like?
4. In what order should we do the work?


## TL;DR

If we want the fastest path to a clean v1, the order should be:

1. Fix the repo skeleton.
2. Fix import and dependency problems.
3. Remove live dependence on `deprecated` and `_pre` files.
4. Restructure namespaces.
5. Add the dataset layer.
6. Rebuild docs/examples/tests around the new structure.

The biggest current blockers are:

- package structure is mid-transition
- runtime dependencies are not declared properly
- some important namespaces fail to import when `harp` is missing
- supported code still depends on `deprecated`
- there is no proper docs/tests/examples/resources layout yet


## What The Dataset Layer Should Do

The dataset layer should be the place where **all data from one session or one trial can live together**.

### Recommended v1 model

- Use plain `xarray.Dataset` as the main container.
- Store the main data inside it as `xarray.DataArray`s.
- Build datasets from outputs of:
  - monosource loaders
  - filetype loaders
  - multisource loaders
  - sync outputs
  - segmentation outputs

### Why this is useful

- it gives the project a single canonical session/trial object
- it makes segmentation of whole sessions easier
- it keeps coordinates and labels explicit
- it fits the existing direction of `DataArray` outputs

### How pandas should still fit in

Pandas is still very useful for readability.

Recommended split:

- canonical storage/computation: `xarray`
- human-readable inspection: `pandas`

So the dataset layer should make it easy to:

- attach or derive readable tables
- inspect event/state tables as pandas
- keep provenance about where each variable came from

### What dataset segmentation should do

Segmenting a dataset should support:

- event-based segmentation
- boolean/state segmentation
- returning multiple trial-level `xr.Dataset`s
- keeping a segment table that explains the slices
- optionally preserving both absolute time and relative segment time


## Current Structure

### Current package structure

```text
src/data_conduit/
  __init__.py
  io/
  datasource/
  multisource/
  globaltimes/
  ttlsync/
  segment/
  timestamps/
  virtualarrays/
  harptools/
  utils/
  validators/
  data_conduit_demos/
  deprecated/
```

### Current top-level workspace shape

```text
repo/
  src/
  dev_stuff/
  1mouse-Poke/
  2mice/
  CNS_Poster_demo_files/
  Old/
  Misc/
  JPVA191B_23102025_run1_g0/
  README.md
  pyproject.toml
  uv.lock
```

### What is currently working reasonably well

Smoke-checked imports/functions that currently look fine in this environment:

- `data_conduit.io`
- `data_conduit.segment`
- `data_conduit.timestamps`
- `data_conduit.globaltimes`
- `data_conduit.ttlsync`
- `data_conduit.virtualarrays`
- `data_conduit.utils`
- `data_conduit.validators`
- wheel/sdist build
- `create_global_clock(...)`
- `index_map_util(...)`
- `collect_file_paths(...)`
- `get_segment(...)`


## Current Problems

### Broken or fragile right now

- `README.md` is empty.
- `pyproject.toml` does not declare the runtime dependencies the code actually uses.
- `src/data_conduit/__init__.py` still exposes only `hello()`.
- `data_conduit.datasource` fails to import if `harp` is missing.
- `data_conduit.multisource` fails to import if `harp` is missing.
- `data_conduit.harptools` fails to import if `harp` is missing.
- `segment` still imports supported functionality from `deprecated`.
- `ttlsync` still uses `_pre` shim files via star imports.
- demos and deprecated content still live inside `src/`.
- there is no proper docs/tests/examples/resources layout.

### Dependency issues to watch

Code currently imports:

- `numpy`
- `pandas`
- `xarray`
- `yaml` / `PyYAML`
- `matplotlib`
- `harp`

Key observations:

- `numpy`, `pandas`, and `xarray` are core dependencies.
- `matplotlib` is mostly optional in spirit.
- `harp` is currently missing in this environment, and that already exposes a namespace-coupling problem.

### Missing files and folders

These should exist and currently do not:

```text
docs/
tests/
tests/unit/
tests/integration/
tests/fixtures/
examples/
resources/
resources/figures/
resources/sample_data/
.github/workflows/
LICENSE
CHANGELOG.md
CONTRIBUTING.md
```


## Deprecated Audit

### What exists in `deprecated`

Code:

- `datasource_core_pre.py`
- `devices_pre.py`
- `filetypes_pre.py`
- `filetypes_pre_2.py`
- `multisource_core_pre.py`
- `multidevice_pre.py`
- `segment_core_pre.py`
- `demo.py`

Notebook/demo material:

- `Demo_workflow.ipynb`
- `Final_demo.ipynb`
- `Placeholder_4Part_Demo.ipynb`
- `claude_demo.ipynb`
- `collect_dfs_demo.ipynb`
- `data_conduit_demo.ipynb`
- `data_conduit_tutorial_and_demo.ipynb`
- `Demos_README.md`

### Everything currently being used that exists in `deprecated`

Direct runtime use found in the current package:

- `segment_boolean_series`
- `slice_event_windows`
- `slice_dataarray_windows`

These are imported from `deprecated.segment_core_pre` by the current `segment` package.

### Deprecated files with clear current replacements

- `datasource_core_pre.py` -> `datasource/datasource_core.py`
- `devices_pre.py` -> `datasource/devices.py`
- `filetypes_pre.py` -> `datasource/filetypes.py`
- `filetypes_pre_2.py` -> `datasource/filetypes.py`
- `multisource_core_pre.py` -> `multisource/multisource_core.py`
- `multidevice_pre.py` -> `multisource/multidevice.py`

### Practical conclusion

- `deprecated` should stay as an archive.
- It should stop being on the live runtime path.
- Before that happens, the three still-used segmentation helpers need to be moved into the real `segment` module.


## Biggest Issues vs Simple Fixes

| Biggest issue | Why it matters | Simple fix now |
| --- | --- | --- |
| Empty README | The package has no public explanation | Write a short real README immediately |
| Missing runtime dependency declarations | Clean installs are misleading/broken | Add actual runtime dependencies to `pyproject.toml` |
| Placeholder root API | The package still looks unfinished at the top level | Replace `hello()` with version + real exports |
| Live use of `deprecated` | Supported code still depends on archived code | Move segmentation helpers into `segment` |
| `_pre` shim modules in `ttlsync` | Harder to document and reason about | Move active code into canonical files |
| Harp import coupling | Too many namespaces fail when `harp` is absent | Make Harp imports lazy/guarded |
| Demos + deprecated under `src/` | Package contents are noisy and confusing | Move them out of `src/` |
| No docs/tests/examples/resources structure | There is nowhere clean to put the next phase of work | Create the folders now |
| Root repo is cluttered | Hard to tell release-critical files from local work | Move assets into `resources/` and establish boundaries |

### Short version

The project is not failing because of one deep algorithmic bug. It is mostly failing because the structure is still transitional.


## Proposed Final Structure

```text
repo/
  docs/
  examples/
  resources/
    figures/
    sample_data/
  tests/
    unit/
    integration/
    fixtures/
  deprecated/
    code_archive/
    notebooks/
  src/data_conduit/
    __init__.py
    datasources/
      __init__.py
      monosource.py
      multisource.py
      devices.py
      filetypes.py
      helpers.py
    datasets/
      __init__.py
      builders.py
      accessors.py
      segmenting.py
      provenance.py
    sync/
      __init__.py
      ttlsync.py
      visualisation.py
    io/
    globaltimes/
    timestamps/
    virtualarrays/
    harptools/
    segment/
    utils/
    validators/
```

### What is missing to get there

- `datasources/`
- `datasets/`
- `sync/`
- migration of live code out of `deprecated` and `_pre`
- docs site
- tests structure
- examples structure
- resources/assets structure
- release/contributor metadata files


## Recommended Order Of Work

### Phase 1. Create the missing project skeleton

- add `docs/`
- add `tests/`
- add `examples/`
- add `resources/`
- add `.github/`
- add `LICENSE`, `CHANGELOG.md`, `CONTRIBUTING.md`

Why first:

- this is cheap
- it immediately makes the repo easier to understand
- it gives stable places for the rest of the work

### Phase 2. Fix imports and dependencies

- declare real runtime dependencies
- stop broad namespace imports from breaking when `harp` is missing
- decide whether Harp functionality is base or optional in practice

### Phase 3. Remove live dependence on old code

- move segmentation helpers out of `deprecated`
- move TTL sync code out of `_pre` files
- freeze `deprecated` as archive-only after extracting anything still needed

### Phase 4. Do the namespace refactor

- `datasource` -> `datasources`
- `DataSource` -> `MonoSource`
- `MultiSource` under `datasources`
- `ttlsync` under `sync`

### Phase 5. Build the dataset layer

- define how session datasets are created
- define naming/provenance rules
- add segmentation on datasets

### Phase 6. Rebuild docs/examples/tests around the final structure

- Sphinx + MyST
- smaller focused examples
- minimal real test suite
- migration guide + changelog


## Action List

### Do soon

- write a real README
- fix `pyproject.toml` dependencies
- replace `hello()` at the package root
- create `docs/`, `tests/`, `examples/`, and `resources/`
- move figures into `resources/figures/`

### Do before renaming packages

- remove direct runtime imports from `deprecated`
- remove `_pre` shims as the canonical implementation path
- make Harp-dependent imports safer
- fix mutable default args and other obvious lint issues

### Do during the main refactor

- add `datasources.monosource`
- add `datasources.multisource`
- add `sync.ttlsync`
- add `datasets`

### Do after the structure is stable

- rebuild API docs cleanly
- split demos into smaller examples
- add minimal real tests
- wire up CI and docs autobuild

## Final note

The key thing to avoid is trying to do docs, datasets, namespace refactors, packaging cleanup, and compatibility fixes all at once. The repo will get cleaner much faster if the early work is structural and dependency-focused.
